# Color defects ("mark on face" / "glue on face") at different resolutions

Extract every image annotated with `mark on face` or `glue on face` (case insensitive) from a COCO file and save it at several resolutions, to study how visible the defect is as the resolution drops.

## Imports

In [1]:
import cv2
import json
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt

## Config

In [2]:
DATA_DIR = Path(r"G:\B1375\DATA\Top")
ANNOTATIONS_FILE = DATA_DIR / "_annotations.coco.json"
OUTPUT_DIR = DATA_DIR / "color_defect_resolutions"

# Category name (case insensitive) -> output folder
TARGET_CATEGORIES = {
    "mark on face": "mark_on_face",
    "glue on face": "glue_on_face",
}

# Real-world size of one pixel in the original photos
ORIGINAL_MM_PER_PX = 0.7

## Resolutions

Each resolution is defined by the real-world size (mm) of one pixel; the scale factor is `ORIGINAL_MM_PER_PX / mm_per_px`.

**Human eye at 1 m**: a normal eye (20/20) resolves ~1 arcminute, i.e. `1000 mm * tan(1/60 deg) ≈ 0.29 mm`. To be able to *see* a feature it needs ~2 samples, so the smallest resolvable detail is ~0.58 mm (one line pair). `eye_1m` below uses 0.29 mm/px as the eye's sampling pitch. Since the originals are 0.7 mm/px (coarser than the eye), this one is an upsample (no new information) and serves as reference; the rest of the resolutions simulate progressively coarser vision / cameras.

In [3]:
EYE_MM_PER_PX = 1000 * math.tan(math.radians(1 / 60))  # 1 arcminute at 1 m
print(f"Eye resolution at 1 m: {EYE_MM_PER_PX:.3f} mm")

RESOLUTIONS_MM_PER_PX = {
    "eye_1m": EYE_MM_PER_PX,
    "0.7mm": 0.7,   # same as original, resaved
    "1.0mm": 1.0,
    "1.4mm": 1.4,
    "2.1mm": 2.1,
    "2.8mm": 2.8,
    "4.2mm": 4.2,
    "5.6mm": 5.6,
}

pd.DataFrame(
    [{"name": n, "mm_per_px": round(mm, 3), "scale": round(ORIGINAL_MM_PER_PX / mm, 3)} for n, mm in RESOLUTIONS_MM_PER_PX.items()]
)

Eye resolution at 1 m: 0.291 mm


,name,mm_per_px,scale
0,eye_1m,0.291,2.406
1,0.7mm,0.700,1.000
2,1.0mm,1.000,0.700
3,1.4mm,1.400,0.500
4,2.1mm,2.100,0.333
5,2.8mm,2.800,0.250
6,4.2mm,4.200,0.167
7,5.6mm,5.600,0.125


## Load annotations and select images

In [4]:
with open(ANNOTATIONS_FILE, encoding="utf-8") as f:
    coco = json.load(f)

categories = {c["id"]: c["name"] for c in coco["categories"]}
print("Categories:", categories)

images = {im["id"]: im for im in coco["images"]}

# folder -> {image_id: [annotations]}
anns_by_folder = {}
for target, folder in TARGET_CATEGORIES.items():
    target_ids = {cid for cid, name in categories.items() if name.strip().lower() == target}
    anns_by_image = {}
    for ann in coco["annotations"]:
        if ann["category_id"] in target_ids:
            anns_by_image.setdefault(ann["image_id"], []).append(ann)
    anns_by_folder[folder] = anns_by_image
    print(f"{len(anns_by_image)} images with '{target}' out of {len(images)}")

Categories: {53: 'Bad trim', 10: 'Broken lipping', 51: 'Damaged face', 56: 'Foil Tear', 14: 'Glue on face', 55: 'Loose Lipping', 57: 'Mark On Face', 6: 'No lipping', 61: 'No paint', 9: 'Short lipping', 8: 'Short skin', 7: 'Slip skin', 13: 'Water Mark'}
43 images with 'mark on face' out of 607
45 images with 'glue on face' out of 607


## Helpers

Annotations use the segmentation polygon when available, otherwise the bbox (`x, y, w, h`).

In [5]:
def ann_to_polygons(ann):
    """Return a list of Nx2 float arrays (original pixel coordinates)."""
    seg = ann.get("segmentation")
    if isinstance(seg, list) and len(seg) > 0 and all(isinstance(p, list) and len(p) >= 6 for p in seg):
        return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg]
    x, y, w, h = ann["bbox"]
    return [np.array([[x, y], [x + w, y], [x + w, y + h], [x, y + h]], dtype=np.float32)]


def resize_image(img, scale):
    h, w = img.shape[:2]
    new_size = (max(1, round(w * scale)), max(1, round(h * scale)))
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, new_size, interpolation=interp)


def draw_annotations(img, anns, scale=1.0, color=(0, 0, 255)):
    out = img.copy()
    for ann in anns:
        for poly in ann_to_polygons(ann):
            cv2.polylines(out, [np.round(poly * scale).astype(np.int32)], True, color, 2)
    return out

## Export

One folder per category in `OUTPUT_DIR` (`mark_on_face`, `glue_on_face`). Per image, in `<category>/<image_stem>/`:
- `original.<ext>`: untouched copy
- `5.6mm.png`: image at 5.6 mm/px
- `max_512.png`, `max_1024.png`, `max_2048.png`: longest side resized to 512 / 1024 / 2048 px
- `<name>_annotated.png`: a copy of each resized image with the defect outline drawn

In [6]:
MM_PER_PX_EXPORT = 5.6
MAX_SIDES = [512, 1024, 2048]

for folder, anns_by_image in anns_by_folder.items():
    for image_id, anns in tqdm(anns_by_image.items(), desc=folder):
        src = DATA_DIR / images[image_id]["file_name"]
        img = cv2.imread(str(src))
        if img is None:
            print(f"Could not read {src}")
            continue

        out_dir = OUTPUT_DIR / folder / src.stem
        out_dir.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(out_dir / f"original{src.suffix}"), img)

        scales = {f"{MM_PER_PX_EXPORT}mm": ORIGINAL_MM_PER_PX / MM_PER_PX_EXPORT}
        scales.update({f"max_{s}": s / max(img.shape[:2]) for s in MAX_SIDES})
        for name, scale in scales.items():
            resized = resize_image(img, scale)
            cv2.imwrite(str(out_dir / f"{name}.png"), resized)
            cv2.imwrite(str(out_dir / f"{name}_annotated.png"), draw_annotations(resized, anns, scale))

glue_on_face: 100%|██████████| 45/45 [13:41<00:00, 18.25s/it]
